# 02 TypeB 上機教材：哪些特徵真的值得放進模型？

**Big HOT：一個看起來有預測力的特徵，如何證明它不是偷看未來、不是偶然、也不是只在訓練期有效？**

這堂課把 Week 2 的策略問題卡轉成資料表、特徵、target 與模型比較。3 小時主線聚焦在：feature engineering、leakage、time split、baseline、模型解讀。


## 課前閱讀與 LMS 回應

課前請先閱讀本資料夾的 `README.md`。

課前 HOT：

```text
If a model uses tomorrow's return as one of today's features,
why might the accuracy look excellent but be useless?
```

LMS 回應格式：

| item | valid / leakage / unclear | reason |
|---|---|---|
| yesterday close |  |  |
| tomorrow return |  |  |
| rolling 20-day volatility using past returns |  |  |
| full-sample mean return |  |  |

英文句：

```text
This feature may cause leakage because ______.
```

課中使用方式：先投影匿名分類結果，再用本 notebook 的 leakage audit 與 model comparison 檢查課前判斷。


## 3 小時 HOT storyline

| 時間 | Cluster | 核心 HOT | 可見產出 |
|---:|---|---|---|
| 0:00-0:25 | C1. Target and timeline | 你的 target 是交易當下可知道的嗎？ | target 定義 |
| 0:25-1:00 | C1. Leakage audit | 哪些特徵偷偷使用未來？ | leakage 分類 |
| 1:00-1:40 | C2. Feature engineering | 金融直覺如何變成 pandas 欄位？ | feature dictionary |
| 1:40-2:20 | C3. Validation | random split 為什麼危險？ | time split |
| 2:20-2:50 | C3. Model comparison | 準確率最高就是最好嗎？ | model comparison table |
| 2:50-3:00 | C4. Evidence handoff | 哪些 evidence 可以帶去 Week 4 回測？ | exit evidence |

**Optional HOT**：feature importance、nonlinear interaction、preprocessing leakage。  
**Appendix HOT**：purged CV、permutation importance、regularization、portfolio estimation error。


## Learning Loop Map（新版 Type B 操作版）

| Loop | Mini-input | BIT / HOT | Visible output | Delayed feedback focus |
|---|---|---|---|---|
| 1 | Target and timeline | Debug + predict | target definition statement | label 在什麼時間點才知道 |
| 2 | Leakage audit | Classify | leakage audit table | shift、future return、full-sample processing |
| 3 | Feature engineering | Transform | feature dictionary | 金融直覺如何變成 pandas 欄位 |
| 4 | Feature stability | Rank/order | feature stability ranking | 圖、統計與金融直覺如何支持排序 |
| 5 | Validation | Compare + predict | split comparison note | random split vs time split |
| 6 | Model evidence | Compare / transform | model choice 與 confusion matrix 解讀 | ML 指標如何翻成交易風險 |


## TypeB 課堂語言

```text
This feature is valid / leakage / unclear because ______.
The model may look better because ______.
I prefer model ___ for trading because ______, even though ______.
The confusion matrix means the trading risk is ______.
```


## 學生回應方式（課中使用）

本堂課每個回答都要連到資料時間線。

| 場景 | 回應格式 |
|---|---|
| Target | `The target is known at ______, so it can / cannot support trading.` |
| Leakage | `This feature is valid / leakage / unclear because ______.` |
| Feature dictionary | `This feature represents ______, but it may fail when ______.` |
| Model choice | `I prefer model ______ because ______, although ______.` |

小組分享時請引用一個表格、圖或 code output。


In [ ]:
# 課堂穩定性設定：預設使用合成 OHLCV 資料。
# 若教室網路穩定且已安裝 yfinance，可以把 USE_ONLINE_DATA 改成 True。
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

SYMBOL = "0050.TW"
USE_ONLINE_DATA = False


def make_synthetic_ohlcv(n=760, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2021-01-01", periods=n)
    t = np.arange(n)
    regime = np.select(
        [t < n * 0.33, t < n * 0.66],
        [0.00045, -0.00015],
        default=0.00025,
    )
    shocks = rng.normal(0, 0.011, n)
    shocks[0] = rng.normal(0, 0.011)
    ret = regime + shocks + 0.08 * np.r_[0, shocks[:-1]]
    close = 100 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.012, n))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.012, n))
    volume = rng.lognormal(mean=15.2, sigma=0.25, size=n) * (1 + 8 * np.abs(ret))
    df = pd.DataFrame(
        {
            "Open": open_,
            "High": high,
            "Low": low,
            "Close": close,
            "Adj Close": close,
            "Volume": volume.astype(int),
        },
        index=dates,
    )
    df.index.name = "Date"
    return df


def load_market_data(symbol=SYMBOL, start="2020-01-01", use_online=USE_ONLINE_DATA):
    if use_online:
        try:
            import yfinance as yf
            df = yf.download(symbol, start=start, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if not df.empty and {"Open", "High", "Low", "Close", "Volume"}.issubset(df.columns):
                print(f"Loaded online data: {symbol}, rows={len(df)}")
                return df.dropna()
        except Exception as exc:
            print("Online download failed; falling back to synthetic data.")
            print(type(exc).__name__, exc)
    print("Using synthetic OHLCV data. Toggle USE_ONLINE_DATA=True for real market data.")
    return make_synthetic_ohlcv()


def add_features(raw):
    df = raw.copy()
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_fwd_1d"] = df["Close"].shift(-1) / df["Close"] - 1
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_gap"] = df["ma_5"] / df["ma_20"] - 1
    df["mom_5"] = df["Close"] / df["Close"].shift(5) - 1
    df["mom_20"] = df["Close"] / df["Close"].shift(20) - 1
    df["vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    volume_mean = df["Volume"].rolling(20).mean()
    volume_std = df["Volume"].rolling(20).std()
    df["volume_z"] = (df["Volume"] - volume_mean) / volume_std
    df = df.dropna()
    df["target_up"] = (df["ret_fwd_1d"] > 0).astype(int)
    return df


def max_drawdown(ret):
    wealth = (1 + ret.fillna(0)).cumprod()
    dd = wealth / wealth.cummax() - 1
    return dd.min()


def sharpe(ret, periods=252):
    vol = ret.std()
    if vol == 0 or np.isnan(vol):
        return np.nan
    return np.sqrt(periods) * ret.mean() / vol


def perf_table(returns_dict):
    rows = []
    for name, ret in returns_dict.items():
        ret = pd.Series(ret).dropna()
        rows.append(
            {
                "strategy": name,
                "ann_return": (1 + ret).prod() ** (252 / len(ret)) - 1 if len(ret) else np.nan,
                "ann_vol": ret.std() * np.sqrt(252),
                "sharpe": sharpe(ret),
                "max_drawdown": max_drawdown(ret),
                "win_rate": (ret > 0).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("strategy").round(4)


raw = load_market_data()
df = add_features(raw)
df.tail()


## C1 HOT 1：target 寫錯，模型會學到什麼？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Debug + predict |
| Think | 60 秒：判斷哪個 target 可以支援交易決策。 |
| Pair/Group | 3 分鐘：兩人說明 target 的時間點。 |
| Visible output | target definition statement。 |
| Delayed feedback | 追問：At what time is this label known? |

HOT 類型：**Debug + Predict**  
請先判斷：`target_up` 應該是今天漲跌、明天漲跌，還是持有期間報酬？


In [ ]:
target_examples = pd.DataFrame(
    [
        ["today_return_up", "Close[t] / Close[t-1] - 1 > 0", "not useful if trading after close"],
        ["next_day_up", "Close[t+1] / Close[t] - 1 > 0", "valid if features use information up to t"],
        ["next_5day_return", "Close[t+5] / Close[t] - 1", "valid but holding period changes"],
        ["max_future_return_20d", "max(Close[t+1:t+20])", "dangerous unless label is explicitly event-based"],
    ],
    columns=["target_name", "definition", "class_discussion"],
)
target_examples


## C1 HOT 2：哪些欄位在偷看未來？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Classify |
| Think | 60 秒：個人先標 valid / leakage / unclear。 |
| Pair/Group | 4 分鐘：小組完成 leakage audit。 |
| Visible output | leakage audit table。 |
| Delayed feedback | 請學生指出哪個 shift 或 preprocessing 造成問題。 |

HOT 類型：**Classify + Justify**  
學生先不跑模型，只做時間線稽核。這一步是金融 ML 的地基。


In [ ]:
leakage_audit = pd.DataFrame(
    {
        "candidate_feature": [
            "ret_1d",
            "mom_20 using Close[t]",
            "vol_20 using returns up to t",
            "ret_fwd_1d",
            "target_up shifted backward by mistake",
            "full-sample z-score",
            "rolling z-score using past 20 days",
        ],
        "student_label": "",
        "reason": "",
    }
)
leakage_audit


## C2 HOT 3：金融直覺如何變成特徵？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Transform |
| Think | 60 秒：先把一個金融直覺寫成欄位名稱。 |
| Pair/Group | 5 分鐘：小組完成 feature dictionary。 |
| Visible output | feature dictionary。 |
| Delayed feedback | 追問：What financial idea does this column represent? |

HOT 類型：**Transform**  
把「趨勢」、「動能」、「波動」、「流動性」各寫成一個欄位，並說明它可能代表什麼金融直覺。


In [ ]:
feature_cols = ["ma_gap", "mom_5", "mom_20", "vol_20", "volume_z", "range_pct"]
feature_dictionary = pd.DataFrame(
    [
        ["ma_gap", "short trend vs medium trend", "trend/momentum", "may lag turning points"],
        ["mom_5", "5-day return", "short momentum", "too noisy"],
        ["mom_20", "20-day return", "momentum", "regime-dependent"],
        ["vol_20", "annualized 20-day volatility", "risk state", "not direction by itself"],
        ["volume_z", "unusual volume", "attention/liquidity", "ambiguous direction"],
        ["range_pct", "high-low range / close", "intraday uncertainty", "may proxy risk"],
    ],
    columns=["feature", "definition", "intuition", "risk"],
)
feature_dictionary


In [ ]:
df[feature_cols + ["target_up"]].describe().T.round(4)


## C2 HOT 4：哪個特徵最可能不穩定？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Rank/order |
| Think | 60 秒：個人排序最不穩定的 feature。 |
| Pair/Group | 3 分鐘：小組用圖形或統計量辯護排序。 |
| Visible output | feature stability ranking。 |
| Delayed feedback | 比較兩組排序，問 evidence 來自圖、統計還是金融直覺。 |

HOT 類型：**Rank + Justify**  
請學生根據圖形與描述統計排序：哪個特徵最可能在不同市場狀態失效？


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True)
for ax, col in zip(axes.ravel(), feature_cols):
    df[col].plot(ax=ax, title=col)
plt.tight_layout()
plt.show()


## C3 HOT 5：random split 為什麼會讓金融模型看起來比較厲害？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare + predict |
| Think | 60 秒：預測 random split 與 time split 哪個分數較高。 |
| Pair/Group | 4 分鐘：小組解釋哪個更接近未來交易。 |
| Visible output | split comparison note。 |
| Delayed feedback | 延後回饋時先問：Which split would you trust for live trading? |

HOT 類型：**Compare + Predict**  
先請學生預測：random split 和 time split 哪一個分數可能比較高？哪一個比較接近未來交易？


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, confusion_matrix

X = df[feature_cols]
y = df["target_up"]

split_idx = int(len(df) * 0.70)
X_train_time, X_test_time = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_time, y_test_time = y.iloc[:split_idx], y.iloc[split_idx:]

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.30, shuffle=True, random_state=7
)

def eval_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, pred),
        "balanced_acc": balanced_accuracy_score(y_test, pred),
        "precision_up": precision_score(y_test, pred, zero_division=0),
        "recall_up": recall_score(y_test, pred, zero_division=0),
    }

baseline_pred = np.repeat(y_train_time.mode().iloc[0], len(y_test_time))
split_compare = pd.DataFrame(
    [
        {
            "split": "time_split_baseline",
            "accuracy": accuracy_score(y_test_time, baseline_pred),
            "balanced_acc": balanced_accuracy_score(y_test_time, baseline_pred),
            "precision_up": precision_score(y_test_time, baseline_pred, zero_division=0),
            "recall_up": recall_score(y_test_time, baseline_pred, zero_division=0),
        },
        {"split": "time_split_logistic", **eval_model(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train_time, y_train_time, X_test_time, y_test_time)},
        {"split": "random_split_logistic", **eval_model(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train_rand, y_train_rand, X_test_rand, y_test_rand)},
    ]
).set_index("split").round(4)
split_compare


## C3 HOT 6：準確率最高的模型，是否一定最適合交易？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare + justify |
| Think | 60 秒：先選你會部署的模型。 |
| Pair/Group | 4 分鐘：小組用 accuracy 以外指標辯護。 |
| Visible output | model choice rationale。 |
| Delayed feedback | 追問：Which error type is more costly in trading? |

HOT 類型：**Compare + Justify**  
請學生同時看 accuracy、balanced accuracy、precision、recall 與可解釋性。


In [ ]:
models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "tree_depth3": DecisionTreeClassifier(max_depth=3, random_state=7),
    "forest_depth4": RandomForestClassifier(n_estimators=200, max_depth=4, random_state=7),
}

rows = []
fitted_models = {}
for name, model in models.items():
    rows.append({"model": name, **eval_model(model, X_train_time, y_train_time, X_test_time, y_test_time)})
    fitted_models[name] = model

model_comparison = pd.DataFrame(rows).set_index("model").round(4)
model_comparison


## C4 HOT 7：confusion matrix 對交易風險代表什麼？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Transform |
| Think | 60 秒：把 false positive / false negative 翻成交易語言。 |
| Pair/Group | 3 分鐘：小組完成錯誤類型解讀。 |
| Visible output | confusion matrix trading interpretation。 |
| Delayed feedback | 請學生比較 ML metric 和 trading risk 的差異。 |

HOT 類型：**Transform**  
把 ML 錯誤改寫成交易語言：false positive 可能是「不該買卻買」，false negative 可能是「錯過機會」。


In [ ]:
chosen_model = fitted_models["forest_depth4"]
pred = chosen_model.predict(X_test_time)
cm = pd.DataFrame(
    confusion_matrix(y_test_time, pred),
    index=["actual_down_or_flat", "actual_up"],
    columns=["pred_down_or_flat", "pred_up"],
)
cm


## C4 HOT 8：故意放入 leakage feature，模型分數會發生什麼事？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Debug + evidence |
| Think | 60 秒：預測加入 leaked feature 後分數會怎樣。 |
| Pair/Group | 3 分鐘：小組解釋為什麼漂亮分數反而危險。 |
| Visible output | leakage evidence note。 |
| Delayed feedback | 教師最後才指出：high score can be invalid evidence。 |

HOT 類型：**Debug + Evidence**  
這題讓學生親眼看到：錯誤資料處理可以讓模型看起來很聰明。


In [ ]:
leaked = df.copy()
leaked["leak_future_return"] = leaked["ret_fwd_1d"]
leak_cols = feature_cols + ["leak_future_return"]

X_leak = leaked[leak_cols]
y_leak = leaked["target_up"]
X_train_leak, X_test_leak = X_leak.iloc[:split_idx], X_leak.iloc[split_idx:]
y_train_leak, y_test_leak = y_leak.iloc[:split_idx], y_leak.iloc[split_idx:]

leak_model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=7)
leak_result = pd.DataFrame(
    [
        {"case": "valid_features", **eval_model(RandomForestClassifier(n_estimators=200, max_depth=4, random_state=7), X_train_time, y_train_time, X_test_time, y_test_time)},
        {"case": "with_leaked_future_return", **eval_model(leak_model, X_train_leak, y_train_leak, X_test_leak, y_test_leak)},
    ]
).set_index("case").round(4)
leak_result


## C4 HOT 9：哪些 evidence 可以帶去 Week 4 回測？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Select + justify |
| Think | 60 秒：個人選出能帶去回測的 evidence。 |
| Pair/Group | 4 分鐘：小組整理 handoff 清單。 |
| Visible output | backtest evidence checklist。 |
| Delayed feedback | 追問：What is still not proven until we backtest? |

HOT 類型：**Select + Justify**  
Week 3 不直接宣布「模型可賺錢」。它只交付可回測的 evidence：模型、特徵、測試期、預測機率。


In [ ]:
proba_model = fitted_models["forest_depth4"]
if hasattr(proba_model, "predict_proba"):
    prob_up = proba_model.predict_proba(X_test_time)[:, 1]
else:
    prob_up = pred

evidence_for_backtest = pd.DataFrame(
    {
        "Close": df.loc[X_test_time.index, "Close"],
        "ret_fwd_1d": df.loc[X_test_time.index, "ret_fwd_1d"],
        "prob_up": prob_up,
        "target_up": y_test_time,
    },
    index=X_test_time.index,
)
evidence_for_backtest.head()


## Optional HOT：3 小時內時間不夠時可跳過

1. **Preprocessing leakage**：standard scaler 應該 fit 在全資料還是訓練資料？
2. **Interpretability trade-off**：logistic 較可解釋，但 forest 分數較高，你選哪個？
3. **Threshold discussion**：若只在 `prob_up > 0.60` 時交易，precision/recall 會怎樣？
4. **Feature removal**：刪掉一個你不信任的特徵，模型分數是否穩定？


In [ ]:
# Optional: threshold changes the meaning of model output.
threshold_table = []
for th in [0.50, 0.55, 0.60, 0.65]:
    pred_th = (evidence_for_backtest["prob_up"] > th).astype(int)
    threshold_table.append(
        {
            "threshold": th,
            "trade_rate": pred_th.mean(),
            "precision_up": precision_score(y_test_time, pred_th, zero_division=0),
            "recall_up": recall_score(y_test_time, pred_th, zero_division=0),
        }
    )
pd.DataFrame(threshold_table).round(4)


## Appendix HOT A1：TimeSeriesSplit 夠不夠？什麼時候需要 purged CV？

HOT 類型：**Compare + Explain**  
如果標籤持有 5 天，訓練與驗證期間太近可能重疊資訊。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=4)
cv_rows = []
for fold, (tr, te) in enumerate(tscv.split(X), start=1):
    cv_rows.append(
        {
            "fold": fold,
            "train_start": X.index[tr[0]].date(),
            "train_end": X.index[tr[-1]].date(),
            "test_start": X.index[te[0]].date(),
            "test_end": X.index[te[-1]].date(),
            "gap_days": (X.index[te[0]] - X.index[tr[-1]]).days,
        }
    )
pd.DataFrame(cv_rows)


## Appendix HOT A2：Feature importance 是答案還是下一個問題？

HOT 類型：**Interpret + Challenge**  
重要特徵不一定有穩定因果；它只是指出下一步該檢查哪裡。


In [ ]:
forest = fitted_models["forest_depth4"]
importances = pd.Series(forest.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind="bar", title="Random forest feature importance")
plt.show()
importances.round(4)


## Appendix HOT A3：非線性互動為什麼可能重要？

HOT 類型：**Transform + Explain**  
動能在低波動時可能有效，高波動時可能失效；這是 tree-based model 常見的吸引力。


In [ ]:
interaction_check = df.assign(
    mom_bucket=pd.qcut(df["mom_20"], 3, labels=["low_mom", "mid_mom", "high_mom"]),
    vol_bucket=pd.qcut(df["vol_20"], 3, labels=["low_vol", "mid_vol", "high_vol"]),
)
interaction_table = interaction_check.pivot_table(
    values="ret_fwd_1d",
    index="mom_bucket",
    columns="vol_bucket",
    aggfunc="mean",
)
interaction_table.round(5)


## Appendix HOT A4：Regularization 與 portfolio estimation error 有什麼關係？

HOT 類型：**Bridge**  
模型限制複雜度，不只是技術細節；在投資組合中，它對應到避免過度相信估計值。


In [ ]:
regularization_bridge = pd.DataFrame(
    [
        ["ML model", "too many flexible parameters", "regularization / depth limit"],
        ["portfolio", "too much weight on noisy expected returns", "weight constraint / shrinkage"],
        ["backtest", "too many tried rules", "holdout / walk-forward / fewer choices"],
    ],
    columns=["context", "overconfidence_problem", "discipline"],
)
regularization_bridge


## Learning Evidence Checklist

本堂課結束前，至少留下這些 evidence：

- [ ] target definition statement。
- [ ] feature dictionary。
- [ ] leakage audit table。
- [ ] random split vs time split interpretation。
- [ ] model choice rationale。
- [ ] confusion matrix trading interpretation。
- [ ] backtest evidence checklist。

Exit ticket：

```text
The feature I trust most is ______ because ______.
The feature I worry about most is ______ because ______.
Before backtesting, we still need ______.
```


## 參考資料

本課程設計參考下列概念來源，重點不是要求學生讀完整篇，而是把研究中的核心判斷轉成上機問題。

- López de Prado, M. (2018). *Advances in Financial Machine Learning*. 用於 financial ML failure、labeling、triple-barrier、meta-labeling、finance cross-validation、backtest overfitting。
- Gu, S., Kelly, B., & Xiu, D. (2020). Empirical Asset Pricing via Machine Learning. *Review of Financial Studies*. 用於 momentum、liquidity、volatility、非線性模型與資產報酬預測。
- Machine Learning and Portfolio Optimization 相關文獻。用於 regularization、cross-validation、estimation error 與 portfolio construction。
- Robust perspective on transaction costs in portfolio optimization 相關技術筆記。用於 transaction cost、turnover、robustness。
- XAI in finance 綜述文獻。用於 feature importance、SHAP、trust、risk assessment、governance。
- Lee, W.-Y. momentum-based sentiment trading strategy 相關研究。用於 Appendix 中 momentum + sentiment、benchmark、transaction cost、long-horizon evaluation。
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 用於 practical ML workflow、train/test、validation、classification metrics、error analysis。
